## Data Download

In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("levshaazz/emg-muscle-recovery")

print("Path to dataset files:", path)

Path to dataset files: /Users/shikaiyi/.cache/kagglehub/datasets/levshaazz/emg-muscle-recovery/versions/1


## Data Preview

In [9]:
import os
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Get data path (uses path from above)
data_path = Path(path)

# Collect all csv and json files
all_files = []
for ext in ['*.csv', '*.json']:
    all_files.extend(data_path.rglob(ext))

# Group by file type (by filename pattern)
file_groups = defaultdict(list)
for f in sorted(all_files):
    rel_path = f.relative_to(data_path)
    if rel_path.name == 'participants.csv':
        file_groups['participants.csv'].append(str(rel_path))
    elif rel_path.name.endswith('_points.json'):
        file_groups['exp_*_points.json'].append(str(rel_path))
    elif rel_path.name.startswith('exp_') and rel_path.name.endswith('.csv'):
        file_groups['exp_*.csv'].append(str(rel_path))

print("=" * 60)
print("Dataset File Overview")
print("=" * 60)
for file_type, paths in sorted(file_groups.items()):
    print(f"\n[{file_type}] {len(paths)} file(s)")
    if len(paths) <= 5:
        for p in paths:
            print(f"  - {p}")
    else:
        for p in paths[:3]:
            print(f"  - {p}")
        print(f"  ... and {len(paths)-3} more")

Dataset File Overview

[exp_*.csv] 126 file(s)
  - data/1/10min/exp_1.csv
  - data/1/10min/exp_2.csv
  - data/1/10min/exp_3.csv
  ... and 123 more

[exp_*_points.json] 123 file(s)
  - data/1/10min/exp_1_points.json
  - data/1/10min/exp_2_points.json
  - data/1/10min/exp_3_points.json
  ... and 120 more

[participants.csv] 1 file(s)
  - participants.csv


In [10]:
# Column names and types for each file type
def get_pandas_dtype_category(dtype):
    """Convert pandas dtype to readable type description"""
    dtype_str = str(dtype)
    if 'int' in dtype_str:
        return 'integer'
    elif 'float' in dtype_str:
        return 'float'
    elif 'object' in dtype_str or 'string' in dtype_str:
        return 'categorical/string'
    elif 'bool' in dtype_str:
        return 'boolean'
    elif 'datetime' in dtype_str:
        return 'datetime'
    return str(dtype)

print("=" * 60)
print("Column Names and Types")
print("=" * 60)

# 1. participants.csv
df_participants = pd.read_csv(data_path / 'participants.csv')
print("\n[participants.csv]")
print("-" * 40)
for col in df_participants.columns:
    dtype_desc = get_pandas_dtype_category(df_participants[col].dtype)
    n_unique = df_participants[col].nunique()
    print(f"  {col:15} -> {dtype_desc} (unique: {n_unique})")

# 2. exp_*.csv (sample)
sample_exp_csv = list(file_groups['exp_*.csv'])[0]
df_emg = pd.read_csv(data_path / sample_exp_csv)
print(f"\n[exp_*.csv] sample: {sample_exp_csv}")
print("-" * 40)
for col in df_emg.columns:
    dtype_desc = get_pandas_dtype_category(df_emg[col].dtype)
    print(f"  {col:15} -> {dtype_desc}")

# 3. exp_*_points.json (sample)
sample_points = list(file_groups['exp_*_points.json'])[0]
with open(data_path / sample_points) as f:
    points = json.load(f)
print(f"\n[exp_*_points.json] sample: {sample_points}")
print("-" * 40)
print("  close_N = timestamp when starting sit-down (bottom of squat, knees bent)")
print("  open_N  = timestamp when standing up (knees extended)")
print("  baseline = reference timestamp before exercise")
print("-" * 40)
for key, val in points.items():
    print(f"  {key:15} -> float (timestamp in seconds)")

Column Names and Types

[participants.csv]
----------------------------------------
  ID              -> integer (unique: 14)
  Gender          -> str (unique: 2)
  Age             -> integer (unique: 8)
  Handiness       -> str (unique: 1)

[exp_*.csv] sample: data/1/10min/exp_1.csv
----------------------------------------
  time            -> float
  emg1            -> float
  emg2            -> float
  emg3            -> float

[exp_*_points.json] sample: data/1/10min/exp_1_points.json
----------------------------------------
  close_N = timestamp when starting sit-down (bottom of squat, knees bent)
  open_N  = timestamp when standing up (knees extended)
  baseline = reference timestamp before exercise
----------------------------------------
  baseline        -> float (timestamp in seconds)
  close_0         -> float (timestamp in seconds)
  open_0          -> float (timestamp in seconds)
  close_1         -> float (timestamp in seconds)
  open_1          -> float (timestamp in sec

In [11]:
# First 5 rows preview
print("[participants.csv] first 5 rows")
display(df_participants.head())

print("\n[exp_*.csv] first 5 rows (EMG signal)")
display(df_emg.head())

# First 5 JSON files preview
print("\n[exp_*_points.json] first 5 files")
points_df = pd.DataFrame([
    json.load(open(data_path / p)) for p in file_groups['exp_*_points.json'][:5]
], index=file_groups['exp_*_points.json'][:5])
display(points_df)

[participants.csv] first 5 rows


,ID,Gender,Age,Handiness
0,1,Male,22,right
1,2,Female,21,right
2,3,Male,21,right
3,4,Male,22,right
4,5,Male,21,right



[exp_*.csv] first 5 rows (EMG signal)


,time,emg1,emg2,emg3
0,0.033366,2.976044,1.189451,0.400513
1,0.049346,1.521465,0.469817,0.190989
2,0.056792,1.521465,0.469817,0.190989
3,0.061424,1.521465,0.469817,0.190989
4,0.061567,1.521465,0.469817,0.190989



[exp_*_points.json] first 5 files


,baseline,close_0,open_0,close_1,open_1,close_2,open_2
data/1/10min/exp_1_points.json,5.194982,5.194987,8.215003,11.127866,14.157181,17.079057,20.108287
data/1/10min/exp_2_points.json,5.205411,5.205415,8.236081,11.158928,14.188139,17.100311,20.129508
data/1/10min/exp_3_points.json,5.203765,5.203770,8.234359,11.157408,14.175607,17.087606,20.106401
data/1/1min/exp_1_points.json,5.191146,5.191150,8.220996,11.144419,14.173022,17.085436,20.114560
data/1/1min/exp_2_points.json,5.205699,5.205703,8.235017,11.157987,14.187257,17.109618,20.139184
